# Implement Top-k Sampling

**Difficulty**: 🟡 Medium

### Problem Statement

Top-k sampling is a decoding strategy that restricts the next-token selection to only the `k` most probable tokens. By zeroing out or masking all tokens outside the top-k, the model avoids sampling from the long tail of unlikely tokens, which can produce incoherent text.

Your goal is to implement top-k sampling from scratch: apply temperature scaling, find the top-k logits, mask everything else to negative infinity, apply softmax to get probabilities, and sample from the resulting distribution.

---

### Requirements

1. **Temperature Scaling**
   - Divide logits by the temperature parameter before filtering.

2. **Top-k Selection**
   - Identify the top-k logits using `torch.topk`.
   - Determine the minimum value among the top-k (the k-th largest logit).

3. **Masking**
   - Set all logits below the k-th largest to `-inf` so they receive zero probability after softmax.

4. **Sampling**
   - Apply softmax to the masked logits to get a valid probability distribution.
   - Sample a token index using `torch.multinomial`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ The output should be a single token index (integer).
- ✅ Only tokens from the top-k set should ever be sampled.
- ✅ Must work with any value of k from 1 to vocab_size.

---

**Companies**: Anthropic, OpenAI, DeepMind, Cohere

---

<details>
  <summary>💡 Hint</summary>

  - `torch.topk(logits, k)` returns the top-k values and their indices.
  - The k-th largest value can be found as `topk_values[..., -1]` (the minimum of the top-k).
  - Use `torch.where(logits < threshold, -inf, logits)` or boolean indexing to mask out non-top-k tokens.
  - After masking, `F.softmax` will assign zero probability to `-inf` positions, giving you a valid distribution over only the top-k tokens.

</details>

---

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# Data generation / setup
torch.manual_seed(42)
vocab_size = 50257

# Create fake logits as if from an LLM's final layer
logits = torch.randn(1, vocab_size)
print(f"Logits shape: {logits.shape}")
print(f"Logits range: [{logits.min().item():.4f}, {logits.max().item():.4f}]")

In [ ]:
def top_k_sample(logits, k=50, temperature=1.0):
    """
    Perform top-k sampling on logits.
    
    Args:
        logits (Tensor): Raw logits of shape (1, vocab_size)
        k (int): Number of top tokens to keep
        temperature (float): Temperature for scaling logits
        
    Returns:
        int: Sampled token index
    """
    # Step 1: Apply temperature scaling
    # TODO: Divide logits by temperature
    ...
    
    # Step 2: Find the top-k logits
    # TODO: Use torch.topk to get top-k values and indices
    ...
    
    # Step 3: Find the threshold (minimum of top-k values)
    # TODO: Get the k-th largest logit value
    ...
    
    # Step 4: Mask all logits below the threshold to -inf
    # TODO: Set logits below threshold to float('-inf')
    ...
    
    # Step 5: Apply softmax to get probabilities
    # TODO: Convert masked logits to probability distribution
    ...
    
    # Step 6: Sample from the distribution
    # TODO: Use torch.multinomial to sample a token
    ...

In [ ]:
# Validation
print("=== Top-k Sampling Validation ===")
print()

# Test 1: Output should be a valid token index
k = 50
token = top_k_sample(logits, k=k, temperature=1.0)
assert isinstance(token, int) or (isinstance(token, torch.Tensor) and token.ndim == 0), \
    f"Output should be a scalar token index, got {type(token)}"
assert 0 <= token < vocab_size, f"Token {token} out of range [0, {vocab_size})"
print(f"Test 1 PASSED: Sampled token {token} is a valid index")

# Test 2: Sampled tokens should only come from the top-k
_, topk_indices = torch.topk(logits, k)
topk_set = set(topk_indices[0].tolist())

num_samples = 1000
samples = [top_k_sample(logits, k=k, temperature=1.0) for _ in range(num_samples)]
samples_set = set(s if isinstance(s, int) else s.item() for s in samples)
outside_topk = samples_set - topk_set
print(f"Top-k set size: {len(topk_set)}")
print(f"Unique tokens sampled: {len(samples_set)}")
assert len(outside_topk) == 0, \
    f"Sampled {len(outside_topk)} tokens outside top-k: {list(outside_topk)[:5]}"
print(f"Test 2 PASSED: All {num_samples} samples came from top-{k} tokens")

# Test 3: k=1 should always return the argmax token
argmax_token = logits.argmax(dim=-1).item()
samples_k1 = [top_k_sample(logits, k=1, temperature=1.0) for _ in range(100)]
all_argmax = all((s if isinstance(s, int) else s.item()) == argmax_token for s in samples_k1)
assert all_argmax, "k=1 should always return the argmax token"
print(f"Test 3 PASSED: k=1 always returns argmax token {argmax_token}")

# Test 4: Different k values
for test_k in [10, 100, 1000]:
    t = top_k_sample(logits, k=test_k, temperature=1.0)
    print(f"Test 4: k={test_k} sampled token {t}")

print("\nAll tests passed!")